### THE SPARK DATA SKEW PROBLEM (No BS Version)

**1. WHAT IS THE PROBLEM?**
It's called "Data Skew". When you run a join, your cluster divides the work. But instead of sharing it evenly, one worker (executor) gets hit with a massive mountain of data and either takes hours to finish or just dies (OOM / Out Of Memory error). Meanwhile, the other workers finish in 2 seconds and sit around doing literally nothing. 

**2. HOW IT ARISES?**
When you join two tables, Spark uses a hash function on the "join key" to group data. The absolute rule of a Spark join is: **ALL rows with the exact same key MUST go to the exact same executor.** If your real-world data isn't perfectly balanced, Spark doesn't care. It blindly shoves all identical keys onto one machine.

**3. SCENARIO / EXAMPLE**
Let's say you join a 100GB `Sales` table with a `Users` table on the `country` column. 
If 80% of your users are from "India", the Spark hash function sends that entire 80GB chunk of "India" rows to ONE single executor. That poor machine gets crushed under 80GB of data, while the other executors get tiny 1GB chunks and finish instantly.

**4. THE ANALOGY**
Imagine a supermarket with 4 cashiers (your executors). 
The manager (Spark) makes a dumb rule based on what people are buying (the join key): "Cashier 1 handles everyone buying Milk. Cashiers 2, 3, and 4 handle everyone buying anything else."
Suddenly, 500 people walk in to buy Milk, and only 3 people buy apples. 
Cashier 1 is completely crushed and quits their job (crashes), while Cashiers 2, 3, and 4 are literally sleeping.

**5. HOW WE FIX IT**
* AQE (Adaptive Query Execution): Keep this ON (standard today). Spark acts like a smart manager—if it sees Cashier 1 dying, it dynamically splits the line and sends the extra people to the empty cashiers mid-job.
* Salting (The manual fix): We hack the join key. We append random numbers to the heavy keys (changing "India" to "India_1", "India_2", "India_3"). This tricks Spark into thinking they are completely different keys, forcing it to spread the data across different cashiers.
* Repartitioning (df.repartition()): If you just need to balance data (not specifically for a join), this is like taking all the rows and dealing them out like a deck of cards so every executor gets the exact same amount.

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
df = spark.read.format('csv')\
    .option("header","True")\
        .option("inferSchema","true")\
            .load('/Volumes/dlt_sk/test_files/test_files/flight_data_download.csv')

df.show()

+---------+-----------+--------------+-------------------+
|flight_id|flight_date|origin_country|destination_country|
+---------+-----------+--------------+-------------------+
|   FL1001| 2026-08-01|            UK|          Singapore|
|   FL1002| 2026-08-02|           USA|                 UK|
|   FL1003| 2026-08-01|         India|                 UK|
|   FL1004| 2026-08-04|     Australia|                USA|
|   FL1005| 2026-08-03|         India|                UAE|
|   FL1006| 2026-08-01|            UK|          Singapore|
|   FL1007| 2026-08-02|     Australia|                USA|
|   FL1008| 2026-08-04|         India|          Australia|
|   FL1009| 2026-08-04|           UAE|                 UK|
|   FL1010| 2026-08-04|           UAE|                 UK|
|   FL1011| 2026-08-01|            UK|          Australia|
|   FL1012| 2026-08-01|           USA|          Australia|
|   FL1013| 2026-08-04|         India|                USA|
|   FL1014| 2026-08-01|           USA|          Singapor

In [0]:
df.count()

25

# SPARK CORE: REPARTITION VS COALESCE (Deep Dive)

## 1. THE ENGINE LEVEL: WIDE VS. NARROW
* **`repartition(n)`: WIDE TRANSFORMATION (Full Shuffle)**
  * **The Bro Logic:** You take all your boxes, dump everything into a massive pile in the front yard, and meticulously repack exactly `n` new boxes so they all weigh the same.
  * **The Jargon:** Forces a full network shuffle. Writes to disk, moves data across the network (Disk & Network I/O). Extremely expensive.
  * **Superpower:** Can **INCREASE** or **DECREASE** the number of partitions.

* **`coalesce(n)`: NARROW TRANSFORMATION (No Shuffle)**
  * **The Bro Logic:** You just walk around the room with duct tape, taping existing small boxes together to make fewer big bundles. You never unpack anything. 
  * **The Jargon:** Merges existing partitions sitting on the same physical machine. Zero network movement. Practically zero cost. Leaves data skewed if it was already skewed.
  * **Limitation:** Can **ONLY DECREASE** the number of partitions.

---

## 2. THE "COALESCE TRAP" (Why jobs get ruined)
* **The Scenario:** You read 1000 files, do some heavy math, and add `.coalesce(10)` at the very end to shrink the output.
* **The Bro Logic:** You think you're using 1000 workers to do the heavy lifting, and then 10 workers to pack it up. Actually, Spark's manager is lazy. It sees the `coalesce(10)` at the end and says, *"Oh, you only want 10 boxes? I'll just hire 10 workers for the WHOLE job."* 
* **The Jargon:** The Catalyst Optimizer pushes the coalesce **upstream**. It forces the read and the heavy transformations to run on just 10 CPU cores, leaving your 990 other cores sitting totally idle.
* **The Fix:** If doing heavy math before shrinking partitions, either:
  1. Force a shuffle using `.repartition(10)` instead.
  2. Break the chain by adding `.cache()` right before you call `.coalesce()`.

---

## 3. THE ARCHITECT'S CHEAT SHEET
| Goal | Command to Use | The "Why" |
| :--- | :--- | :--- |
| **Wake up idle CPU cores?** | `repartition(n)` | `coalesce` literally cannot increase partitions. |
| **Balance skewed/lopsided data?** | `repartition(n)` | `coalesce` tapes boxes blindly, ignoring weight. |
| **Shrink files right before writing?** | `coalesce(n)` | Avoids a massive, expensive network shuffle at the finish line (assuming no heavy math is left). |

In [0]:
# it has been blocked by databricks for all shared and serverless clusters for security purposes 
df.rdd.getNumPartitions()

---------------------------------------------------------------------------
PySparkNotImplementedError                Traceback (most recent call last)
File <command-6531045569821705>, line 2
      1 # it has been blocked by databricks for all shared and serverless clusters for security purposes 
----> 2 df.rdd.getNumPartitions()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2373, in DataFrame.rdd(self)
   2371 @property
   2372 def rdd(self) -> "RDD[Row]":
-> 2373     raise PySparkNotImplementedError(
   2374         errorClass="NOT_IMPLEMENTED",
   2375         messageParameters={"feature": "rdd"},
   2376     )

PySparkNotImplementedError: [NOT_IMPLEMENTED] Using custom code using PySpark RDDs is not allowed on serverless compute. We suggest using mapInPandas or mapInArrow for the most common use cases, or switch to Dedicated access mode if you require RDDs. For more details on compatibility and limitations, check: https://docs.databricks.com

In [0]:
import pyspark.sql.functions as F

# This tells Spark to look at the partition ID for every row, 
# find the unique ones, and count them. 
# It works perfectly on Serverless.
num_partitions = df.select(F.spark_partition_id()).distinct().count()

print(f"Number of partitions: {num_partitions}")

Number of partitions: 1


# THE SPARK WRITE PARADOX: BINS, FILES, AND 2026 DATABRICKS FIXES

## 1. WHY WE MAKE "BINS" ON DISK (`.partitionBy`)
* **The Bro Logic:** If everything is dumped into one massive folder, finding "Japan" sales means digging through 1 billion rows of global trash. 
* **The Jargon:** This is a **"Full Table Scan."** It's incredibly slow.
* **The Fix:** We use `.partitionBy("country")` to create separate folders on S3/Azure.
* **The Goal:** **"Partition Pruning."** Tomorrow, when we query `WHERE country='Japan'`, Spark ignores 98% of the data and only opens the Japan folder. The query takes 2 seconds instead of 45 minutes.

---

## 2. THE SMALL FILES DISASTER (The Naive Write)
* **The Scenario:** You use `df.repartition(10)`. Your 10 workers (executors) now have perfectly balanced RAM, but every worker holds a random mix of all countries.
* **The Bro Logic:** 10 workers walk up to the 4 country bins. Because they all have mixed data, every single worker drops a tiny file into every single bin. You get 40 tiny files. Multiply this by thousands of workers, and you get millions of tiny files.
* **The Jargon:** High **"Metadata Overhead"** and slow cloud API calls. Your next read query will take 30 minutes just opening the files before doing any actual math.

---

## 3. THE OOM PARADOX (Trying to be too smart)
* **The Scenario:** You try to fix small files using `df.repartition(10, "country")`.
* **The Bro Logic:** You force the workers to trade papers so one guy gets 100% of the India data, so he can write ONE massive file. 
* **The Catch (Data Skew):** If India is 80% of your total data, that one worker gets absolutely crushed under the heavy weight and quits his job. 
* **The Jargon:** Hash Partitioning causes severe **"Data Skew"**, leading to an **Out Of Memory (OOM)** crash on the executor.

---

## 4. THE 2026 DATABRICKS FIX (Let the Engine Do the Heavy Lifting)
* Stop doing manual `repartition("country")` to fix small files. It's an outdated legacy Spark practice.
* **The Databricks Way:** We use Delta Lake's built-in engine optimizations.
* **The Code:**
  ```python
  spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
  spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

In [0]:
new_df = df.repartition(4)

In [0]:
num_partitions = new_df.select(F.spark_partition_id()).distinct().count()
print(f"Number of partitions: {num_partitions}")

Number of partitions: 4


In [0]:
new_df.withColumn("partitionId",spark_partition_id()).groupBy("partitionId").count().show()

+-----------+-----+
|partitionId|count|
+-----------+-----+
|          0|    6|
|          1|    7|
|          2|    6|
|          3|    6|
+-----------+-----+



In [0]:
new_df  = df.repartition(30)

In [0]:
new_df.withColumn("partitionId",spark_partition_id()).groupBy("partitionId").count().show()

+-----------+-----+
|partitionId|count|
+-----------+-----+
|          0|    1|
|          1|    1|
|          2|    1|
|          3|    1|
|          9|    1|
|         10|    1|
|         11|    1|
|         12|    1|
|         13|    1|
|         14|    1|
|         15|    1|
|         16|    1|
|         17|    1|
|         18|    1|
|         19|    1|
|         20|    1|
|         21|    1|
|         22|    1|
|         23|    1|
|         24|    1|
+-----------+-----+
only showing top 20 rows


In [0]:
new_df  = df.repartition(3,"destination_country")

In [0]:
new_df.withColumn("partitionId",spark_partition_id()).groupBy("partitionId").count().show()

+-----------+-----+
|partitionId|count|
+-----------+-----+
|          0|   20|
|          1|    5|
+-----------+-----+



In [0]:
new_df  = df.repartition(5)

In [0]:
new_df.withColumn("partitionId",spark_partition_id()).groupBy("partitionId").count().show()

+-----------+-----+
|partitionId|count|
+-----------+-----+
|          0|    5|
|          1|    5|
|          2|    5|
|          3|    5|
|          4|    5|
+-----------+-----+



In [0]:
cls_df = new_df.coalesce(3)

In [0]:
cls_df.withColumn("partitionId",spark_partition_id()).groupBy("partitionId").count().show()

+-----------+-----+
|partitionId|count|
+-----------+-----+
|          0|    5|
|          1|   10|
|          2|   10|
+-----------+-----+

